# 📖 Beyond the Cover Analytics: Inteligência Editorial com NLP & LLMs
## Desafio Técnico & Negocial A3Data para Editora
---
### Contexto do Desafio:
A editora contratou a **A3Data** para explorar uma base histórica com mais de **2 milhões de avaliações de livros** e **207.000 títulos únicos catalogados**.
- **Gargalo Atual**: 5 analistas dedicados em tempo integral, levando cerca de 3 dias úteis por análise exploratória manual (120 horas-homem).
- **Custo Salarial da Equipe**: R$ 5.000,00 por analista = **R$ 25.000,00/mês** (R$ 300.000,00/ano).
- **Objetivos Centrais**:
  1. **Consolidação & Higienização**: Tratamento de títulos variantes (ex: *The Hobbit* e variações padronizados para autor 'J. R. R. Tolkien' e categoria 'Ficção Juvenil') e deduplicação estrita de reviews.
  2. **Panorama Geral do Mercado**: Tendências temporais e distribuições filtráveis por período (anos).
  3. **Performance por Autor**: Comparativo por faixas de volume de reviews, séries temporais individuais, extração contextual de termos (15+ palavras) e Relatório Executivo Editorial irreverente e afiado.
  4. **Análise Estratégica de Gêneros**: Séries temporais históricas de tração de mercado (volume) e satisfação (nota média), além de matriz de índices (*Paixão de Nicho*, *Polêmica/Hype* e *Consistência*).
  5. **Mesa-Redonda Editorial & Radar de Entrevistas**: Consenso unificado entre personas e diretoria, e seleção de leitores qualificados pelos 4 pilares analíticos.
  6. **Business Case & ROI**: Redução drástica de tempo de análise e retorno financeiro comprovado.

In [ ]:
# 1. Configuração do ambiente e imports
import os
import sys
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona o diretório src ao path
sys.path.insert(0, os.path.abspath('../src'))
from data_loader import DataLoader
from nlp_engine import NLPEngine
from llm_engine import LLMEngine
from business_metrics import ROICalculator

sns.set_theme(style='whitegrid')
print('✅ Ambiente analítico configurado com sucesso!')

### 2. Visão Macro do Catálogo & Verificação de 'The Hobbit'
Verificamos os KPIs gerais e a integridade da consolidação canônica de *The Hobbit* e seu autor *J. R. R. Tolkien*.

In [ ]:
loader = DataLoader(data_dir='../data')
kpis = loader.get_metricas_gerais()

print(f'📚 Total de Títulos Únicos: {kpis["total_livros"]:,}')
print(f'⭐ Total de Avaliações Consolidadas: {kpis["total_avaliacoes"]:,}')
print(f'👤 Total de Leitores Únicos Ativos: {kpis["total_usuarios"]:,}')
print(f'⭐ Nota Média Global: {kpis["media_score_global"]:.2f} / 5.0')

# Verificação canônica de The Hobbit e Tolkien
hobbit_check = loader.conn.execute("""
    SELECT b.Title, b.authors, b.categories, count(r.Id) as total_reviews, round(avg(r.score), 2) as media
    FROM books b
    LEFT JOIN ratings r ON b.norm_title_key = r.norm_title_key
    WHERE b.norm_title_key = 'the hobbit'
    GROUP BY b.Title, b.authors, b.categories;
""").df()
print('\nConsolidação Canônica de The Hobbit:')
display(hobbit_check)

### 3. Tendências Temporais e Filtro de Período do Mercado (1996 a 2013)
Demonstramos a distribuição global de estrelas e o Top 10 gêneros filtrados por período de anos (requisito de negócio).

In [ ]:
# Exemplo com filtro de período: 2005 a 2010
ano_inicio, ano_fim = 2005, 2010
df_estrelas_periodo = loader.get_distribuicao_estrelas(ano_min=ano_inicio, ano_max=ano_fim)
df_top_gen_periodo = loader.get_top_generos_geral(limit=10, ano_min=ano_inicio, ano_max=ano_fim)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=df_estrelas_periodo, x='Estrelas', y='Total Avaliações', palette='Blues_r', ax=ax1)
ax1.set_title(f'Distribuição de Notas ({ano_inicio}-{ano_fim})')
ax1.set_xlabel('Estrelas')
ax1.set_ylabel('Volume')

sns.barplot(data=df_top_gen_periodo, x='Total Avaliações', y='Gênero', palette='viridis', ax=ax2)
ax2.set_title(f'Top 10 Gêneros por Volume de Leitura ({ano_inicio}-{ano_fim})')
ax2.set_xlabel('Total de Avaliações')

plt.tight_layout()
plt.show()

### 4. Performance por Autor: Filtro por Faixas de Volume de Avaliações
Comparação justa de autores por faixas de porte de reviews (`0 a 1.000`, `1.001 a 5.000`, `5.001 a 10.000`, `Mais de 10.000`).

In [ ]:
# Teste na faixa de 1.001 a 5.000 avaliações
df_melhores_faixa, df_piores_faixa = loader.get_ranking_geral_autores(faixa_volume='1.001 a 5.000', limit=8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=df_melhores_faixa, x='Nota Média', y='Autor', palette='Greens_r', ax=ax1)
ax1.set_title('Top 8 Autores Mais Bem Avaliados (1.001 a 5.000 reviews)')
ax1.set_xlim(3.8, 5.0)

sns.barplot(data=df_piores_faixa, x='Nota Média', y='Autor', palette='Reds', ax=ax2)
ax2.set_title('Top 8 Autores com Menor Média (1.001 a 5.000 reviews)')
ax2.set_xlim(1.0, 4.2)

plt.tight_layout()
plt.show()

### 5. Investigação Detalhada por Autor: NLP (15+ Termos) & Relatório Executivo Afiado
Análise aprofundada de J. R. R. Tolkien (com todas as 8.543 reviews e 3.648 de *The Hobbit*), extração de termos recorrentes e síntese editorial.

In [ ]:
autor_foco = 'J. R. R. Tolkien'
df_tolkien_reviews = loader.get_todas_reviews_autor(autor_foco)
nlp = NLPEngine()
metricas_tolkien = nlp.calcular_metricas_autor(df_tolkien_reviews)

print(f'Autor em Estudo: {autor_foco}')
print(f'Total de Avaliações Consolidadas: {metricas_tolkien["total_avaliacoes"]:,}')
print(f'Nota Média: ⭐ {metricas_tolkien["nota_media"]:.2f} / 5.0')
print(f'🟢 Resenhas Positivas: {metricas_tolkien["pct_positivas"]:.1f}% ({metricas_tolkien["total_positivas"]:,})')
print(f'🔴 Resenhas Críticas: {metricas_tolkien["pct_negativas"]:.1f}% ({metricas_tolkien["total_negativas"]:,})')
print(f'\n🏷️ Termos Recorrentes em Elogios (15+ termos):\n{metricas_tolkien["top_termos_elogio"]}')
print(f'\n🏷️ Termos Recorrentes em Críticas (15+ termos):\n{metricas_tolkien["top_termos_critica"]}')

# Geração do Relatório Executivo Editorial irreverente e sem jargões engessados
llm = LLMEngine()
relatorio_editorial = llm.gerar_sintese_executiva_detalhada_autor(
    autor_foco,
    metricas_tolkien,
    df_reviews=df_tolkien_reviews
)
print('\n' + '='*70)
print(relatorio_editorial)
print('='*70)

### 6. Análise Estratégica de Gêneros: Séries Temporais Instrutivas & Matriz de Índices
Gráficos focados em decisões para a editora: Tração de Mercado (volume anual) e Satisfação do Leitor (nota média anual).

In [ ]:
df_ts_generos = loader.get_series_temporais_generos()
generos_foco = ['Ficção', 'Ficção Juvenil', 'Biografia & Autobiografia', 'Religião']
df_ts_sub = df_ts_generos[df_ts_generos['genero'].isin(generos_foco)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for g in generos_foco:
    sub = df_ts_sub[df_ts_sub['genero'] == g]
    ax1.plot(sub['ano'], sub['total_reviews'], marker='o', label=g)
    ax2.plot(sub['ano'], sub['score_medio'], marker='s', label=g)

ax1.set_title('1. Tração de Mercado: Volume Anual de Leituras por Gênero')
ax1.set_xlabel('Ano')
ax1.set_ylabel('Volume de Reviews')
ax1.legend()

ax2.set_title('2. Satisfação do Leitor: Nota Média Anual por Gênero')
ax2.set_xlabel('Ano')
ax2.set_ylabel('Nota Média (1 a 5)')
ax2.set_ylim(3.6, 4.8)
ax2.legend()

plt.tight_layout()
plt.show()

# Matriz de Índices Proprietários
df_indices = loader.get_metricas_generos(limit=10)
print('\nMatriz Estratégica de Gêneros & Índices Proprietários:')
display(df_indices[['Gênero', 'Total Avaliações', 'Nota Média', 'Desvio Padrão', 'Índice Paixão de Nicho', 'Índice Polêmica / Hype', 'Consistência de Agradecimento']])

### 7. Mesa-Redonda Estratégica: Conselho Editorial & Consenso Unificado
Simulação do debate entre as 3 Personas Literárias (Marta, Diego e Helena), o CEO e o CFO.

In [ ]:
debate = llm.simular_debate_conselho_editorial(autor_foco, df_tolkien_reviews)

print('--- DIÁLOGOS EM PLENÁRIA DO CONSELHO EDITORIAL ---')
for f in debate['dialogo']:
    print(f"{f['personagem']} ({f['papel']}): {f['fala']}\n")

print('--- CONSENSO UNIFICADO FORMALIZADO ---')
for k, v in debate['consenso'].items():
    print(f'{k.upper()}: {v}')

### 8. Radar de Entrevistas com Leitores: Scoreboard dos 4 Pilares Analíticos
Identificação dos leitores mais qualificados para entrevistas em profundidade.

In [ ]:
df_scoreboard = loader.get_scoreboard_entrevistas(top_n=10)
print('Top 10 Leitores Qualificados pelo Índice Ajustado (0 a 100):')
display(df_scoreboard[[
    'profileName', 'indice_ajustado', 'n_reviews', 'n_livros',
    'perspicacia_pct', 'perolas_pct', 'tendencias_pct', 'versatilidade_pct', 'especialista_pico'
]])

### 9. Business Case & Análise de Retorno sobre Investimento (ROI)
Comparativo financeiro entre a metodologia manual e a plataforma automatizada.

In [ ]:
calc = ROICalculator()
relatorio_impacto = calc.gerar_relatorio_impacto_executivo()
print(relatorio_impacto)
